In [1]:
import pandas as pd
import numpy as np
from numba import njit
from time import time

In [2]:
@njit()
def kemeny_dist_paralle(base_rankings, candidate_ranking):
    nb_voters=base_rankings.shape[0]
    nb_candidates=base_rankings.shape[1]
    #print(f'voter;{nb_voters},candidates:{nb_candidates}')
    kemeny_dist=0
    for i in range(nb_voters):
        tau=0
        base_ranking=base_rankings[i]
        #if i==0:
        #print(f'base ranking : {show}')
        for j in range(nb_candidates):
            for k in range(j+1,nb_candidates):
                #print(f'i:{j},j:{k}')
                #print(f'a_i:{candidate_ranking[j]}')
                #print(f'a_k:{candidate_ranking[k]}')
                #print(f'pref:{-np.sign((base_ranking[j] - base_ranking[k]))}')
                tau =tau+ int((np.sign(candidate_ranking[j] - candidate_ranking[k])==-np.sign(base_ranking[j] - base_ranking[k])))
                #print(f'kt:{tau}')
        #print(f'voter:{i},tau:{tau}')
        kemeny_dist+=tau
    #print(f'kemeny distance:{kemeny_dist}')    
    kemeny_dist=kemeny_dist/nb_voters
    return kemeny_dist

def compute_kemeny_distance_parallel_greedy(batch_base_rankings,final_rankings):
    bsz=batch_base_rankings.shape[0]
    kemeny_distances=np.empty(bsz)
    start=time()

    for i in range(bsz):
        #print(f'i,{i}')
        base_rankings=batch_base_rankings[i]
        #print(f'base rankings shape: {base_rankings.shape}')
        final_ranking=final_rankings[i]
        #print(f'final rankings shape: {final_ranking.shape}')
        start=time()
        kemeny_distance=kemeny_dist_paralle(base_rankings,final_ranking)
        #print(f'type kemeny distance,{type(kemeny_distance)},i:{i}')
        kemeny_distances[i]=kemeny_distance
    #print(f'kemeny distance parallel computing time:{time.time()-start},distance parallel:{kemeny_distances[0:10]}')
    #print(f'distance parallel:{kemeny_distances[0:10]},type_ distances:{type(kemeny_distances)}')
    return  kemeny_distances

def permutation_to_ranking_greedy(batch_permutation):
    ranking=np.argsort(batch_permutation,axis=1)
    return ranking

In [3]:
nation_rankings_df=pd.read_csv("nation_rankings.csv",skiprows=12,sep=",",)
nation_rankings_df

,Country,population_ranking,hdi_ranking,gpi_ranking,cpi_ranking,gii_ranking,di_ranking,le_ranking,gpd_ranking,schooling_ranking,wtv_ranking,wla_ranking
0,Afghanistan,34.0,88.0,98.0,96.0,88.0,82.0,95.0,77.0,93.0,84.0,33.0
1,Algeria,32.0,47.0,69.0,52.0,77.0,64.0,40.0,48.0,63.0,48.0,9.0
2,Angola,40.0,67.0,25.0,75.0,89.0,66.0,89.0,64.0,74.0,58.0,17.0
3,Argentina,29.0,24.0,13.0,35.0,46.0,23.0,29.0,27.0,28.0,39.0,7.0
4,Australia,51.0,3.0,11.0,4.0,14.0,3.0,2.0,12.0,11.0,19.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Venezuela,47.0,54.0,78.0,98.0,99.0,86.0,49.0,98.0,44.0,80.0,26.0
96,Vietnam,14.0,45.0,27.0,55.0,24.0,80.0,36.0,35.0,54.0,13.0,52.0
97,Yemen,43.0,93.0,97.0,97.0,87.0,95.0,74.0,99.0,84.0,74.0,38.0
98,Zambia,61.0,74.0,15.0,61.0,78.0,55.0,91.0,79.0,62.0,75.0,31.0


In [4]:
idx_to_nation_dict = nation_rankings_df["Country"].to_dict()
idx_to_nation_dict

{0: 'Afghanistan',
 1: 'Algeria',
 2: 'Angola',
 3: 'Argentina',
 4: 'Australia',
 5: 'Austria',
 6: 'Azerbaijan',
 7: 'Bangladesh',
 8: 'Belarus',
 9: 'Belgium',
 10: 'Benin',
 11: 'Bolivia',
 12: 'Brazil',
 13: 'Bulgaria',
 14: 'Burkina Faso',
 15: 'Burundi',
 16: 'Cambodia',
 17: 'Cameroon',
 18: 'Canada',
 19: 'Chad',
 20: 'Chile',
 21: 'China',
 22: 'Colombia',
 23: 'Cuba',
 24: 'Czech Republic',
 25: 'Dominican Republic',
 26: 'Ecuador',
 27: 'Egypt',
 28: 'Ethiopia',
 29: 'France',
 30: 'Germany',
 31: 'Ghana',
 32: 'Greece',
 33: 'Guatemala',
 34: 'Guinea',
 35: 'Haiti',
 36: 'Honduras',
 37: 'Hungary',
 38: 'India',
 39: 'Indonesia',
 40: 'Iran',
 41: 'Iraq',
 42: 'Israel',
 43: 'Italy',
 44: 'Ivory Coast',
 45: 'Japan',
 46: 'Jordan',
 47: 'Kazakhstan',
 48: 'Kenya',
 49: 'Laos',
 50: 'Madagascar',
 51: 'Malawi',
 52: 'Malaysia',
 53: 'Mali',
 54: 'Mexico',
 55: 'Morocco',
 56: 'Mozambique',
 57: 'Myanmar',
 58: 'Nepal',
 59: 'Netherlands',
 60: 'Nicaragua',
 61: 'Niger',
 62

In [5]:
base_rankings_input=nation_rankings_df.drop(columns=["Country","wla_ranking","le_ranking","schooling_ranking"]).to_numpy().T
# base_rankings_input=nation_rankings_df.drop(columns=["Country","le_ranking","schooling_ranking","wla_ranking"]).to_numpy().T
base_rankings_input.shape,base_rankings_input

((8, 100),
 array([[34., 32., 40., 29., 51., 91., 84.,  7., 88., 75., 71., 74.,  5.,
         98., 54., 72., 67., 50., 36., 65., 59.,  0., 27., 77., 80., 78.,
         62., 13., 11., 20., 17., 44., 81., 64., 69., 76., 85., 86.,  1.,
          3., 15., 33., 92., 22., 49., 10., 79., 60., 25., 97., 48., 58.,
         42., 55.,  9., 37., 45., 24., 46., 63., 99., 52.,  6.,  4., 87.,
         41., 12., 35., 83., 57.,  8., 70., 38., 66., 96., 95., 23., 26.,
         28., 53., 82., 93., 56., 89., 21., 18., 94., 73., 16., 30., 31.,
         90., 19.,  2., 39., 47., 14., 43., 61., 68.],
        [88., 47., 67., 24.,  3.,  9., 44., 61., 30.,  5., 87., 56., 38.,
         27., 95., 96., 71., 73.,  6., 98., 23., 34., 41., 43., 17., 40.,
         48., 49., 90., 13.,  1., 66., 18., 63., 89., 79., 64., 22., 62.,
         53., 37., 58., 12., 16., 82., 11., 46., 28., 69., 65., 91., 86.,
         31., 97., 42., 57., 92., 68., 70.,  4., 59., 99., 83., 84., 77.,
         39., 55., 19., 21., 26., 29., 78., 20

In [6]:
base_rankings_input

array([[34., 32., 40., 29., 51., 91., 84.,  7., 88., 75., 71., 74.,  5.,
        98., 54., 72., 67., 50., 36., 65., 59.,  0., 27., 77., 80., 78.,
        62., 13., 11., 20., 17., 44., 81., 64., 69., 76., 85., 86.,  1.,
         3., 15., 33., 92., 22., 49., 10., 79., 60., 25., 97., 48., 58.,
        42., 55.,  9., 37., 45., 24., 46., 63., 99., 52.,  6.,  4., 87.,
        41., 12., 35., 83., 57.,  8., 70., 38., 66., 96., 95., 23., 26.,
        28., 53., 82., 93., 56., 89., 21., 18., 94., 73., 16., 30., 31.,
        90., 19.,  2., 39., 47., 14., 43., 61., 68.],
       [88., 47., 67., 24.,  3.,  9., 44., 61., 30.,  5., 87., 56., 38.,
        27., 95., 96., 71., 73.,  6., 98., 23., 34., 41., 43., 17., 40.,
        48., 49., 90., 13.,  1., 66., 18., 63., 89., 79., 64., 22., 62.,
        53., 37., 58., 12., 16., 82., 11., 46., 28., 69., 65., 91., 86.,
        31., 97., 42., 57., 92., 68., 70.,  4., 59., 99., 83., 84., 77.,
        39., 55., 19., 21., 26., 29., 78., 20., 85., 32., 94., 51.,  8

In [7]:
base_rankings_input.shape

(8, 100)

In [11]:
from kemenyOptimalTransformerTest import *

In [12]:
config_file="args.conf"

In [14]:
import json
from argparse import Namespace

try:
        with open(config_file, 'r') as f:
            config_args_dict = json.load(f)

        # Convert config dict to an object (like argparse.Namespace)
        # so train() can access args with dot notation (e.g., args.lr)
        class ConfigObject:
            def __init__(self, **entries):
                self.__dict__.update(entries)

            def __getattr__(self, name):
                # Return None if arg is not in config, instead of raising error
                return self.__dict__.get(name, None)

        args= ConfigObject(**config_args_dict)
        
except FileNotFoundError:
    print(f"Error: Configuration file not found at {config_file}")
    exit(1)
except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from {config_file}")
    exit(1)


In [15]:
args.checkpoint_dir

'kemeny_transformer_checkpoint/various_input_kemeny_transformer/linear_embedding/100_voters_200_items_kemeny_transformer_without_guide'

In [16]:
args.checkpoint_file="checkpoint_epoch_995.pkl"
args.checkpoint_file

'checkpoint_epoch_995.pkl'

In [17]:
checkpoint_path=args.checkpoint_dir+"/"+args.checkpoint_file 
checkpoint_path

'kemeny_transformer_checkpoint/various_input_kemeny_transformer/linear_embedding/100_voters_200_items_kemeny_transformer_without_guide/checkpoint_epoch_995.pkl'

In [ ]:
from kemenyTransformer import *


In [15]:
args = DotDict()

args.nb_candidates = 100
args.bsz = 1024
args.dim_emb = 128
args.dim_ff = 512
args.dim_input_candidates = 8
args.nb_layers_encoder = 3
args.nb_layers_decoder = 2
args.nb_heads = 8
args.nb_epochs =500
args.nb_batch_per_epoch = 500
args.gpu_id ='0'
args.lr = 1e-4
args.tol = 1e-3
args.baseline_alpha=0.05
args.beam_size=1000
args.roll_out_bsz=1024
args.nb_batch_test = 20
args.batchnorm = True  # if batchnorm=True  than batch norm is used
#args.batchnorm = False # if batchnorm=False than layer norm is used
args.max_len_PE = 1000
checkpoint_file = "kemeny_transformer_checkpoint/fine_tuning_repeat_jiggling/checkpoint_24-04-10--10-22-52-n100-gpu0,1.pkl"
os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu_id)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print('GPU name: {:s}, gpu_id: {:s}'.format(torch.cuda.get_device_name(0),args.gpu_id))
print(f'device: { device}')

GPU name: NVIDIA RTX 4500 Ada Generation, gpu_id: 0
device: cuda


In [16]:
checkpoint_file

'kemeny_transformer_checkpoint/fine_tuning_repeat_jiggling/checkpoint_24-04-10--10-22-52-n100-gpu0,1.pkl'

In [17]:
model_baseline=initial_model_multi_to_single(args,checkpoint_file,device)
model_baseline.eval()

GPU name: NVIDIA RTX 4500 Ada Generation, gpu_id: 0
cuda
1
Epoch: 365, tot_time_ckpt: 4.094day



kemeny_transformer_test(
  (input_emb): Linear(in_features=8, out_features=128, bias=True)
  (encoder): Transformer_encoder(
    (MHA_layers): ModuleList(
      (0-2): 3 x MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
      )
    )
    (linear1_layers): ModuleList(
      (0-2): 3 x Linear(in_features=128, out_features=512, bias=True)
    )
    (linear2_layers): ModuleList(
      (0-2): 3 x Linear(in_features=512, out_features=128, bias=True)
    )
    (norm1_layers): ModuleList(
      (0-2): 3 x BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (norm2_layers): ModuleList(
      (0-2): 3 x BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (decoder): Transformer_decoder(
    (decoder_layers): ModuleList(
      (0): AutoRegressiveDecoderLayer(
        (Wq_selfatt): Linear(in_features=128, out_features=128, bias=True)
        (Wk_selfat

In [18]:
test_dataset_val=torch.from_numpy(base_rankings_input).unsqueeze(0).transpose(1,2)
test_dataset_val.shape

torch.Size([1, 100, 8])

In [19]:
test_dataset_val=torch.from_numpy(base_rankings_input).unsqueeze(0).transpose(1,2).to(device,dtype=torch.float32)
start=time()
with torch.no_grad():
    final_rankings_greedy_perm,_,scores_greedy,_=model_baseline(test_dataset_val,args.beam_size,greedy=True,beam_search=False)
print(f'inference time:{time()-start}')

inference time:0.4060516357421875


In [20]:
final_rankings_greedy_perm,final_rankings_greedy_perm.shape

(tensor([[30, 92, 81, 18, 59, 45, 80, 93,  4, 29, 77, 43, 21,  9,  5, 78, 42, 24,
          52, 67, 68, 91, 20, 32, 38, 37, 12, 70, 72,  3, 69, 13, 88, 85, 54, 96,
          76, 39, 66, 22, 90, 65, 47,  8, 74, 40, 55, 27, 87, 26, 79, 25,  7, 31,
           1, 46, 84, 48, 63, 94,  6, 58, 33, 11,  2, 36, 73, 44, 57, 98, 89, 62,
          41, 16, 28, 64, 17, 49, 50, 71, 23, 51, 99, 56, 14, 34, 10, 86, 53, 35,
          60, 83,  0, 61, 95, 97, 75, 15, 19, 82]], device='cuda:0'),
 torch.Size([1, 100]))

In [21]:
final_rankings_greedy_perm=final_rankings_greedy_perm.detach().cpu().numpy()
final_rankings_greedy_perm[0]

array([30, 92, 81, 18, 59, 45, 80, 93,  4, 29, 77, 43, 21,  9,  5, 78, 42,
       24, 52, 67, 68, 91, 20, 32, 38, 37, 12, 70, 72,  3, 69, 13, 88, 85,
       54, 96, 76, 39, 66, 22, 90, 65, 47,  8, 74, 40, 55, 27, 87, 26, 79,
       25,  7, 31,  1, 46, 84, 48, 63, 94,  6, 58, 33, 11,  2, 36, 73, 44,
       57, 98, 89, 62, 41, 16, 28, 64, 17, 49, 50, 71, 23, 51, 99, 56, 14,
       34, 10, 86, 53, 35, 60, 83,  0, 61, 95, 97, 75, 15, 19, 82])

In [22]:
final_rankings_greedy=permutation_to_ranking_greedy(final_rankings_greedy_perm)[0]
final_rankings_greedy

array([92, 54, 64, 29,  8, 14, 60, 52, 43, 13, 86, 63, 26, 31, 84, 97, 73,
       76,  3, 98, 22, 12, 39, 80, 17, 51, 49, 47, 74,  9,  0, 53, 23, 62,
       85, 89, 65, 25, 24, 37, 45, 72, 16, 11, 67,  5, 55, 42, 57, 77, 78,
       81, 18, 88, 34, 46, 83, 68, 61,  4, 90, 93, 71, 58, 75, 41, 38, 19,
       20, 30, 27, 79, 28, 66, 44, 96, 36, 10, 15, 50,  6,  2, 99, 91, 56,
       33, 87, 48, 32, 70, 40, 21,  1,  7, 59, 94, 35, 95, 69, 82])

In [23]:
final_rankings_greedy_name=[idx_to_nation_dict[i] for i in final_rankings_greedy_perm[0]]
final_rankings_greedy_name

['Germany',
 'United Kingdom',
 'Switzerland',
 'Canada',
 'Netherlands',
 'Japan',
 'Sweden',
 'United States',
 'Australia',
 'France',
 'South Korea',
 'Italy',
 'China',
 'Belgium',
 'Austria',
 'Spain',
 'Israel',
 'Czech Republic',
 'Malaysia',
 'Poland',
 'Portugal',
 'United Arab Emirates',
 'Chile',
 'Greece',
 'India',
 'Hungary',
 'Brazil',
 'Russia',
 'Saudi Arabia',
 'Argentina',
 'Romania',
 'Bulgaria',
 'Turkey',
 'Thailand',
 'Mexico',
 'Vietnam',
 'South Africa',
 'Indonesia',
 'Philippines',
 'Colombia',
 'Ukraine',
 'Peru',
 'Kazakhstan',
 'Belarus',
 'Serbia',
 'Iran',
 'Morocco',
 'Egypt',
 'Tunisia',
 'Ecuador',
 'Sri Lanka',
 'Dominican Republic',
 'Bangladesh',
 'Ghana',
 'Algeria',
 'Jordan',
 'Tanzania',
 'Kenya',
 'Pakistan',
 'Uzbekistan',
 'Azerbaijan',
 'Nepal',
 'Guatemala',
 'Bolivia',
 'Angola',
 'Honduras',
 'Senegal',
 'Ivory Coast',
 'Myanmar',
 'Zambia',
 'Uganda',
 'Nigeria',
 'Iraq',
 'Cambodia',
 'Ethiopia',
 'Papua New Guinea',
 'Cameroon',
 'La

In [24]:
final_rankings_greedy_kemeny_distance_transformer=kemeny_dist_paralle(base_rankings_input,final_rankings_greedy)

In [25]:
final_rankings_greedy_kemeny_distance_transformer

974.25

In [26]:
start=time()
with torch.no_grad():
    _,final_rankings_beam_perm,_,scores_beam=model_baseline(test_dataset_val,args.beam_size,greedy=False,beam_search=True)
print(f'inference time:{time()-start}')

inference time:0.518688440322876


In [27]:
final_rankings_beam_perm

tensor([[[30, 92, 81,  ..., 15, 19, 82],
         [30, 92, 81,  ..., 15, 19, 82],
         [30, 92, 81,  ..., 15, 19, 82],
         ...,
         [30, 92, 81,  ..., 19, 15, 82],
         [30, 92, 81,  ..., 19, 15, 82],
         [30, 92, 81,  ..., 19, 15, 82]]], device='cuda:0')

In [28]:
final_rankings_beam_perm_cpu=final_rankings_beam_perm.detach().cpu().numpy()
final_rankings_beam_perm_cpu

array([[[30, 92, 81, ..., 15, 19, 82],
        [30, 92, 81, ..., 15, 19, 82],
        [30, 92, 81, ..., 15, 19, 82],
        ...,
        [30, 92, 81, ..., 19, 15, 82],
        [30, 92, 81, ..., 19, 15, 82],
        [30, 92, 81, ..., 19, 15, 82]]])

In [29]:
final_rankings_beam=permutation_to_ranking_greedy(final_rankings_beam_perm_cpu[0])
final_rankings_beam

array([[92, 54, 64, ..., 95, 69, 82],
       [92, 54, 64, ..., 95, 69, 82],
       [92, 54, 64, ..., 95, 69, 82],
       ...,
       [92, 54, 64, ..., 95, 69, 82],
       [92, 54, 64, ..., 95, 68, 82],
       [92, 54, 64, ..., 95, 69, 82]])

In [30]:
kenemy_distances_beam=[]
for final_ranking_beam in final_rankings_beam:
    kenemy_distance_beam=kemeny_dist_paralle(base_rankings_input,final_ranking_beam)
    print(kenemy_distance_beam)
    kenemy_distances_beam.append(kenemy_distance_beam)


974.0
974.25
974.5
974.0
974.25
974.25
974.0
974.5
974.25
974.25
974.75
974.5
974.0
974.0
974.0
974.5
974.25
974.0
974.5
973.75
974.25
974.25
974.0
974.0
974.5
974.0
974.25
974.75
974.25
974.0
974.5
975.0
974.5
974.25
974.25
974.5
974.25
974.0
974.0
973.75
974.75
974.5
974.25
974.0
974.25
974.0
974.5
974.25
974.75
974.75
974.25
974.0
973.75
974.25
974.0
974.5
974.5
974.0
974.25
974.25
974.5
974.25
974.25
974.25
974.0
974.0
974.0
974.75
974.25
974.5
974.0
974.0
974.5
974.25
973.75
975.0
974.5
974.0
974.25
974.5
974.25
974.75
974.75
975.25
974.5
974.5
974.5
974.25
974.25
974.25
974.75
974.5
974.0
974.5
973.75
974.75
974.0
974.5
975.0
974.25
974.25
973.75
974.5
974.0
974.25
974.25
974.25
974.0
974.75
974.75
974.25
974.5
974.5
974.0
973.75
974.0
974.25
974.0
974.5
974.0
974.0
974.5
974.0
974.0
974.25
973.75
974.25
974.25
975.0
974.25
974.5
974.25
974.0
974.75
974.75
974.5
974.5
973.75
974.5
974.0
974.25
974.25
974.5
974.75
974.25
974.25
974.5
974.5
974.5
974.5
974.0
974.0
974.5
973.75
974.

In [31]:
minimal_kemeny_distance_beam_idx=np.where(kenemy_distances_beam == np.min(kenemy_distances_beam))
minimal_kemeny_distance_beam_idx

(array([409, 616, 807]),)

In [32]:
kenemy_distances_beam[minimal_kemeny_distance_beam_idx[0][0]]

973.5

In [33]:
final_rankings_beam_name=[idx_to_nation_dict[i] for i in final_rankings_beam_perm_cpu[0][minimal_kemeny_distance_beam_idx[0][0]]]
final_rankings_beam_name

['Germany',
 'United Kingdom',
 'Switzerland',
 'Canada',
 'Netherlands',
 'Sweden',
 'Japan',
 'United States',
 'Australia',
 'France',
 'South Korea',
 'Italy',
 'China',
 'Belgium',
 'Austria',
 'Spain',
 'Israel',
 'Czech Republic',
 'Malaysia',
 'Poland',
 'Portugal',
 'United Arab Emirates',
 'Chile',
 'Greece',
 'India',
 'Hungary',
 'Brazil',
 'Russia',
 'Saudi Arabia',
 'Argentina',
 'Romania',
 'Bulgaria',
 'Turkey',
 'Thailand',
 'Mexico',
 'Vietnam',
 'South Africa',
 'Indonesia',
 'Philippines',
 'Colombia',
 'Ukraine',
 'Peru',
 'Kazakhstan',
 'Belarus',
 'Serbia',
 'Iran',
 'Morocco',
 'Egypt',
 'Tunisia',
 'Ecuador',
 'Sri Lanka',
 'Dominican Republic',
 'Bangladesh',
 'Ghana',
 'Algeria',
 'Jordan',
 'Tanzania',
 'Kenya',
 'Pakistan',
 'Uzbekistan',
 'Azerbaijan',
 'Bolivia',
 'Nepal',
 'Guatemala',
 'Angola',
 'Honduras',
 'Senegal',
 'Ivory Coast',
 'Myanmar',
 'Zambia',
 'Uganda',
 'Nigeria',
 'Iraq',
 'Cambodia',
 'Ethiopia',
 'Papua New Guinea',
 'Laos',
 'Camero

In [34]:
from gurobi_test import *

In [35]:
start=time()
kemeny_ranking_gurobi,running_time_gurobi=aggregate_kemeny(8,100,base_rankings_input)
print(f'inference time:{time()-start}')

Set parameter Username
Academic license - for non-commercial use only - expires 2026-06-30
1
2
3
4
5
inference time:12.163447380065918


In [36]:
kemeny_ranking_gurobi

array([91., 55., 71., 23.,  4., 11., 65., 47., 44., 13., 85., 61., 29.,
       26., 79., 96., 73., 80.,  2., 98., 21., 14., 38., 74., 17., 51.,
       52., 45., 69.,  9.,  1., 50., 24., 63., 87., 88., 66., 25., 27.,
       30., 48., 72., 16., 12., 64.,  8., 56., 41., 57., 75., 78., 82.,
       18., 86., 31., 49., 83., 70., 59.,  3., 90., 92., 68., 62., 76.,
       40., 39., 19., 22., 28., 32., 81., 36., 60., 42., 94., 37., 10.,
       15., 53.,  5.,  0., 99., 93., 54., 33., 89., 46., 34., 77., 43.,
       20.,  6.,  7., 58., 95., 35., 97., 67., 84.])

In [37]:
kemeny_ranking_gurobi_perm=np.argsort(kemeny_ranking_gurobi)
kemeny_ranking_gurobi_perm

array([81, 30, 18, 59,  4, 80, 92, 93, 45, 29, 77,  5, 43,  9, 21, 78, 42,
       24, 52, 67, 91, 20, 68,  3, 32, 37, 13, 38, 69, 12, 39, 54, 70, 85,
       88, 96, 72, 76, 22, 66, 65, 47, 74, 90,  8, 27, 87,  7, 40, 55, 31,
       25, 26, 79, 84,  1, 46, 48, 94, 58, 73, 11, 63, 33, 44,  6, 36, 98,
       62, 28, 57,  2, 41, 16, 23, 49, 64, 89, 50, 14, 17, 71, 51, 56, 99,
       10, 53, 34, 35, 86, 60,  0, 61, 83, 75, 95, 15, 97, 19, 82])

In [38]:
kemeny_ranking_gurobi_name=[idx_to_nation_dict[i] for i in kemeny_ranking_gurobi_perm]
kemeny_ranking_gurobi_name

['Switzerland',
 'Germany',
 'Canada',
 'Netherlands',
 'Australia',
 'Sweden',
 'United Kingdom',
 'United States',
 'Japan',
 'France',
 'South Korea',
 'Austria',
 'Italy',
 'Belgium',
 'China',
 'Spain',
 'Israel',
 'Czech Republic',
 'Malaysia',
 'Poland',
 'United Arab Emirates',
 'Chile',
 'Portugal',
 'Argentina',
 'Greece',
 'Hungary',
 'Bulgaria',
 'India',
 'Romania',
 'Brazil',
 'Indonesia',
 'Mexico',
 'Russia',
 'Thailand',
 'Turkey',
 'Vietnam',
 'Saudi Arabia',
 'South Africa',
 'Colombia',
 'Philippines',
 'Peru',
 'Kazakhstan',
 'Serbia',
 'Ukraine',
 'Belarus',
 'Egypt',
 'Tunisia',
 'Bangladesh',
 'Iran',
 'Morocco',
 'Ghana',
 'Dominican Republic',
 'Ecuador',
 'Sri Lanka',
 'Tanzania',
 'Algeria',
 'Jordan',
 'Kenya',
 'Uzbekistan',
 'Nepal',
 'Senegal',
 'Bolivia',
 'Pakistan',
 'Guatemala',
 'Ivory Coast',
 'Azerbaijan',
 'Honduras',
 'Zambia',
 'Nigeria',
 'Ethiopia',
 'Myanmar',
 'Angola',
 'Iraq',
 'Cambodia',
 'Cuba',
 'Laos',
 'Papua New Guinea',
 'Uganda',

In [39]:
final_rankings_kemeny_distance_gurobi=kemeny_dist_paralle(base_rankings_input,kemeny_ranking_gurobi)
final_rankings_kemeny_distance_gurobi

965.5

In [40]:
kemeny_ranking_gurobi.shape

(100,)

In [41]:
import markov_chain as mc


In [42]:
start=time()
kemeny_ranking_mc=mc.aggregate_rank_mc(base_rankings_input)
print(f'inference time:{time()-start}')

inference time:0.04528093338012695


In [43]:
kemeny_ranking_mc

array([91, 57, 64, 23,  8, 14, 74, 41, 56, 13, 82, 66, 19, 33, 80, 99, 77,
       85,  3, 98, 20, 15, 39, 70, 18, 53, 49, 43, 62,  6,  0, 45, 37, 68,
       83, 92, 71, 34, 22, 26, 55, 73, 25,  9, 65,  2, 58, 44, 59, 84, 75,
       79, 16, 86, 27, 47, 81, 69, 61,  4, 94, 90, 54, 52, 78, 40, 36, 17,
       21, 24, 30, 72, 38, 60, 51, 89, 32, 11, 12, 48, 10,  7, 97, 95, 50,
       28, 93, 46, 31, 76, 42, 35,  1,  5, 67, 88, 29, 96, 63, 87])

In [44]:
final_rankings_kemeny_distance_mc=kemeny_dist_paralle(base_rankings_input,kemeny_ranking_mc)
final_rankings_kemeny_distance_mc,

(997.5,)

In [45]:
kemeny_ranking_mc_perm=np.argsort(kemeny_ranking_mc)
kemeny_ranking_mc_perm

array([30, 92, 45, 18, 59, 93, 29, 81,  4, 43, 80, 77, 78,  9,  5, 21, 52,
       67, 24, 12, 20, 68, 38,  3, 69, 42, 39, 54, 85, 96, 70, 88, 76, 13,
       37, 91, 66, 32, 72, 22, 65,  7, 90, 27, 47, 31, 87, 55, 79, 26, 84,
       74, 63, 25, 62, 40,  8,  1, 46, 48, 73, 58, 28, 98,  2, 44, 11, 94,
       33, 57, 23, 36, 71, 41,  6, 50, 89, 16, 64, 51, 14, 56, 10, 34, 49,
       17, 53, 99, 95, 75, 61,  0, 35, 86, 60, 83, 97, 82, 19, 15])

In [46]:
kemeny_ranking_mc_name=[idx_to_nation_dict[i] for i in kemeny_ranking_mc_perm]
kemeny_ranking_mc_name

['Germany',
 'United Kingdom',
 'Japan',
 'Canada',
 'Netherlands',
 'United States',
 'France',
 'Switzerland',
 'Australia',
 'Italy',
 'Sweden',
 'South Korea',
 'Spain',
 'Belgium',
 'Austria',
 'China',
 'Malaysia',
 'Poland',
 'Czech Republic',
 'Brazil',
 'Chile',
 'Portugal',
 'India',
 'Argentina',
 'Romania',
 'Israel',
 'Indonesia',
 'Mexico',
 'Thailand',
 'Vietnam',
 'Russia',
 'Turkey',
 'South Africa',
 'Bulgaria',
 'Hungary',
 'United Arab Emirates',
 'Philippines',
 'Greece',
 'Saudi Arabia',
 'Colombia',
 'Peru',
 'Bangladesh',
 'Ukraine',
 'Egypt',
 'Kazakhstan',
 'Ghana',
 'Tunisia',
 'Morocco',
 'Sri Lanka',
 'Ecuador',
 'Tanzania',
 'Serbia',
 'Pakistan',
 'Dominican Republic',
 'Nigeria',
 'Iran',
 'Belarus',
 'Algeria',
 'Jordan',
 'Kenya',
 'Senegal',
 'Nepal',
 'Ethiopia',
 'Zambia',
 'Angola',
 'Ivory Coast',
 'Bolivia',
 'Uzbekistan',
 'Guatemala',
 'Myanmar',
 'Cuba',
 'Honduras',
 'Rwanda',
 'Iraq',
 'Azerbaijan',
 'Madagascar',
 'Uganda',
 'Cambodia',
 'P

In [47]:
import KwikSort as ks

In [48]:
vertices=np.arange(0,100)
vertices

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [49]:
base_rankings_perm=np.argsort(base_rankings_input,axis=1)
base_rankings_perm

array([[21, 38, 93, 39, 63, 12, 62,  7, 70, 54, 45, 28, 66, 27, 96, 40,
        88, 30, 85, 92, 29, 84, 43, 76, 57, 48, 77, 22, 78,  3, 89, 90,
         1, 41,  0, 67, 18, 55, 72, 94,  2, 65, 52, 97, 31, 56, 58, 95,
        50, 44, 17,  4, 61, 79, 14, 53, 82, 69, 51, 20, 47, 98, 26, 59,
        33, 19, 73, 16, 99, 34, 71, 10, 15, 87, 11,  9, 35, 23, 25, 46,
        24, 32, 80, 68,  6, 36, 37, 64,  8, 83, 91,  5, 42, 81, 86, 75,
        74, 49, 13, 60],
       [81, 30, 80,  4, 59,  9, 18, 92, 77,  5, 93, 45, 42, 29, 91, 78,
        43, 24, 32, 67, 72, 68, 37, 20,  3, 88, 69, 13, 47, 70,  8, 52,
        74, 85, 21, 90, 79, 40, 12, 65, 25, 22, 54, 23,  6, 96, 46,  1,
        26, 27, 87, 76, 94, 39, 95, 66, 11, 55, 41, 60, 83,  7, 38, 33,
        36, 49, 31,  2, 57, 48, 58, 16, 99, 17, 98, 89, 82, 64, 71, 35,
        86, 84, 44, 62, 63, 73, 51, 10,  0, 34, 28, 50, 56, 97, 75, 14,
        15, 53, 19, 61],
       [13, 52, 81, 24,  5, 43, 18, 30, 68, 59, 69,  4,  9,  3, 78, 98,
        92, 80

In [50]:
nation_rankings_df.iloc[30]

Country               Germany
population_ranking       17.0
hdi_ranking               1.0
gpi_ranking               7.0
cpi_ranking               3.0
gii_ranking               5.0
di_ranking                5.0
le_ranking               13.0
gpd_ranking               3.0
schooling_ranking         0.0
wtv_ranking               2.0
wla_ranking              49.0
Name: 30, dtype: object

In [51]:
start=time()
kemeny_ranking_perm_ks=ks.KwikSort(vertices,base_rankings_input)
print(f'inference time:{time()-start}')

inference time:0.05811142921447754


In [52]:
kemeny_ranking_ks=np.argsort(np.array(kemeny_ranking_perm_ks))
kemeny_ranking_ks

array([90, 57, 61, 22,  4, 11, 62, 43, 45, 16, 83, 63, 27, 25, 71, 95, 68,
       78,  1, 97, 17,  9, 29, 74, 18, 49, 50, 41, 72,  8,  0, 47, 23, 65,
       85, 86, 75, 24, 26, 31, 58, 76, 12, 13, 69,  7, 66, 48, 52, 77, 80,
       81, 19, 84, 32, 46, 87, 59, 64,  2, 89, 91, 55, 56, 79, 30, 38, 20,
       21, 28, 33, 73, 39, 67, 51, 98, 40, 10, 14, 53, 15,  3, 99, 92, 54,
       34, 93, 42, 35, 82, 44, 36,  5,  6, 60, 94, 37, 96, 70, 88])

In [53]:
kemeny_distance_ks=kemeny_dist_paralle(base_rankings_input,kemeny_ranking_ks)
kemeny_distance_ks

989.0

In [54]:
kemeny_ranking_ks_perm=np.argsort(kemeny_ranking_ks)
kemeny_ranking_ks_name=[idx_to_nation_dict[i] for i in kemeny_ranking_ks_perm]
kemeny_ranking_ks_name

['Germany',
 'Canada',
 'Netherlands',
 'Switzerland',
 'Australia',
 'United Kingdom',
 'United States',
 'Japan',
 'France',
 'China',
 'South Korea',
 'Austria',
 'Israel',
 'Italy',
 'Spain',
 'Sweden',
 'Belgium',
 'Chile',
 'Czech Republic',
 'Malaysia',
 'Poland',
 'Portugal',
 'Argentina',
 'Greece',
 'Hungary',
 'Bulgaria',
 'India',
 'Brazil',
 'Romania',
 'Colombia',
 'Peru',
 'Indonesia',
 'Mexico',
 'Russia',
 'Thailand',
 'Turkey',
 'United Arab Emirates',
 'Vietnam',
 'Philippines',
 'Saudi Arabia',
 'South Africa',
 'Egypt',
 'Tunisia',
 'Bangladesh',
 'Ukraine',
 'Belarus',
 'Morocco',
 'Ghana',
 'Kazakhstan',
 'Dominican Republic',
 'Ecuador',
 'Serbia',
 'Kenya',
 'Sri Lanka',
 'Tanzania',
 'Nigeria',
 'Pakistan',
 'Algeria',
 'Iran',
 'Myanmar',
 'Uzbekistan',
 'Angola',
 'Azerbaijan',
 'Bolivia',
 'Nepal',
 'Guatemala',
 'Jordan',
 'Senegal',
 'Cambodia',
 'Ivory Coast',
 'Zambia',
 'Burkina Faso',
 'Ethiopia',
 'Rwanda',
 'Cuba',
 'Honduras',
 'Iraq',
 'Laos',
 'C

In [55]:
from heuristic_consensus_ranking import HeuristicConsensusRanker 

In [56]:
base_rankings_input

array([[34., 32., 40., 29., 51., 91., 84.,  7., 88., 75., 71., 74.,  5.,
        98., 54., 72., 67., 50., 36., 65., 59.,  0., 27., 77., 80., 78.,
        62., 13., 11., 20., 17., 44., 81., 64., 69., 76., 85., 86.,  1.,
         3., 15., 33., 92., 22., 49., 10., 79., 60., 25., 97., 48., 58.,
        42., 55.,  9., 37., 45., 24., 46., 63., 99., 52.,  6.,  4., 87.,
        41., 12., 35., 83., 57.,  8., 70., 38., 66., 96., 95., 23., 26.,
        28., 53., 82., 93., 56., 89., 21., 18., 94., 73., 16., 30., 31.,
        90., 19.,  2., 39., 47., 14., 43., 61., 68.],
       [88., 47., 67., 24.,  3.,  9., 44., 61., 30.,  5., 87., 56., 38.,
        27., 95., 96., 71., 73.,  6., 98., 23., 34., 41., 43., 17., 40.,
        48., 49., 90., 13.,  1., 66., 18., 63., 89., 79., 64., 22., 62.,
        53., 37., 58., 12., 16., 82., 11., 46., 28., 69., 65., 91., 86.,
        31., 97., 42., 57., 92., 68., 70.,  4., 59., 99., 83., 84., 77.,
        39., 55., 19., 21., 26., 29., 78., 20., 85., 32., 94., 51.,  8

In [57]:
hcr=HeuristicConsensusRanker(base_rankings_input)

In [58]:
start=time()
max_agreement_consensus_ranking_perm=hcr.maximize_agreement()
print(f'inference time:{time()-start}')
max_agreement_consensus_ranking=np.argsort(np.array(max_agreement_consensus_ranking_perm))
max_agreement_consensus_ranking_kemeny_distance=kemeny_dist_paralle(base_rankings_input,max_agreement_consensus_ranking)
max_agreement_consensus_ranking_kemeny_distance,max_agreement_consensus_ranking

inference time:0.059053897857666016


(998.25,
 array([95, 53, 65, 19,  7, 16, 69, 49, 54, 12, 82, 60, 21, 32, 80, 99, 75,
        86,  3, 98, 20, 23, 36, 73, 17, 50, 44, 47, 72,  4,  0, 51, 30, 62,
        83, 91, 70, 24, 25, 29, 55, 68, 26,  5, 67,  2, 57, 41, 56, 81, 77,
        79, 14, 87, 33, 43, 84, 64, 61,  6, 93, 90, 63, 58, 76, 31, 37, 13,
        18, 22, 39, 74, 40, 66, 45, 85, 27, 10,  8, 46, 15, 11, 97, 94, 52,
        28, 92, 48, 35, 78, 38, 42,  1,  9, 59, 89, 34, 96, 71, 88]))

In [59]:
kemeny_ranking_max_agreement_name=[idx_to_nation_dict[i] for i in max_agreement_consensus_ranking_perm]
kemeny_ranking_max_agreement_name

['Germany',
 'United Kingdom',
 'Japan',
 'Canada',
 'France',
 'Italy',
 'Netherlands',
 'Australia',
 'Spain',
 'United States',
 'South Korea',
 'Switzerland',
 'Belgium',
 'Poland',
 'Malaysia',
 'Sweden',
 'Austria',
 'Czech Republic',
 'Portugal',
 'Argentina',
 'Chile',
 'Brazil',
 'Romania',
 'China',
 'Hungary',
 'India',
 'Israel',
 'South Africa',
 'Thailand',
 'Indonesia',
 'Greece',
 'Peru',
 'Bulgaria',
 'Mexico',
 'Vietnam',
 'Turkey',
 'Colombia',
 'Philippines',
 'Ukraine',
 'Russia',
 'Saudi Arabia',
 'Kazakhstan',
 'United Arab Emirates',
 'Morocco',
 'Ecuador',
 'Serbia',
 'Sri Lanka',
 'Egypt',
 'Tunisia',
 'Bangladesh',
 'Dominican Republic',
 'Ghana',
 'Tanzania',
 'Algeria',
 'Belarus',
 'Iran',
 'Kenya',
 'Jordan',
 'Pakistan',
 'Uzbekistan',
 'Bolivia',
 'Nepal',
 'Guatemala',
 'Nigeria',
 'Myanmar',
 'Angola',
 'Senegal',
 'Ivory Coast',
 'Iraq',
 'Azerbaijan',
 'Honduras',
 'Zambia',
 'Ethiopia',
 'Cuba',
 'Rwanda',
 'Cambodia',
 'Papua New Guinea',
 'Madaga

In [60]:
start=time()
min_regret_consensus_ranking_perm=hcr.minimize_regret()
print(f'inference time:{time()-start}')
min_regret_consensus_ranking=np.argsort(np.array(min_regret_consensus_ranking_perm))
min_regret_consensus_ranking_kemeny_distance=kemeny_dist_paralle(base_rankings_input,min_regret_consensus_ranking)
min_regret_consensus_ranking_kemeny_distance,min_regret_consensus_ranking

inference time:0.005506753921508789


(999.75,
 array([91, 53, 65, 20,  7, 16, 67, 49, 56, 12, 86, 59, 21, 37, 77, 97, 72,
        80,  3, 98, 18, 22, 36, 85, 17, 48, 42, 51, 74,  4,  0, 45, 31, 60,
        84, 88, 68, 24, 29, 27, 63, 73, 28,  6, 61,  2, 55, 41, 54, 79, 76,
        83, 15, 87, 33, 43, 78, 70, 57,  5, 90, 92, 69, 58, 75, 32, 34, 13,
        19, 23, 39, 82, 40, 62, 46, 94, 26, 10,  9, 44, 14, 11, 99, 93, 50,
        25, 89, 47, 35, 71, 38, 52,  1,  8, 64, 95, 30, 96, 66, 81]))

In [61]:
kemeny_ranking_min_regret_name=[idx_to_nation_dict[i] for i in min_regret_consensus_ranking_perm]
kemeny_ranking_min_regret_name

['Germany',
 'United Kingdom',
 'Japan',
 'Canada',
 'France',
 'Netherlands',
 'Italy',
 'Australia',
 'United States',
 'Spain',
 'South Korea',
 'Switzerland',
 'Belgium',
 'Poland',
 'Sweden',
 'Malaysia',
 'Austria',
 'Czech Republic',
 'Chile',
 'Portugal',
 'Argentina',
 'Brazil',
 'China',
 'Romania',
 'Hungary',
 'Thailand',
 'South Africa',
 'Indonesia',
 'Israel',
 'India',
 'Vietnam',
 'Greece',
 'Peru',
 'Mexico',
 'Philippines',
 'Turkey',
 'Colombia',
 'Bulgaria',
 'Ukraine',
 'Russia',
 'Saudi Arabia',
 'Kazakhstan',
 'Ecuador',
 'Morocco',
 'Sri Lanka',
 'Ghana',
 'Serbia',
 'Tunisia',
 'Dominican Republic',
 'Bangladesh',
 'Tanzania',
 'Egypt',
 'United Arab Emirates',
 'Algeria',
 'Kenya',
 'Jordan',
 'Belarus',
 'Nepal',
 'Pakistan',
 'Bolivia',
 'Guatemala',
 'Ivory Coast',
 'Senegal',
 'Iran',
 'Uzbekistan',
 'Angola',
 'Zambia',
 'Azerbaijan',
 'Honduras',
 'Nigeria',
 'Myanmar',
 'Uganda',
 'Cambodia',
 'Iraq',
 'Ethiopia',
 'Papua New Guinea',
 'Madagascar',
 '

EA result

In [62]:
nation_decor_result_df=pd.read_csv("nation_data/nation_decor_result.csv").to_numpy()
nation_decor_result_df

array([[62.   , 37.   , 42.   , 19.   ,  7.   ,  8.   , 44.   , 35.   ,
        38.   ,  9.   , 57.   , 41.   , 17.   , 18.   , 55.   , 67.   ,
        46.   , 55.   ,  5.   , 67.   , 16.   , 15.   , 25.   , 53.   ,
        12.   , 35.   , 32.   , 34.   , 51.   ,  5.   ,  1.   , 35.   ,
        17.   , 41.   , 57.   , 59.   , 43.   , 16.   , 17.   , 24.   ,
        35.   , 50.   , 14.   ,  9.   , 42.   ,  5.   , 38.   , 28.   ,
        38.   , 50.   , 52.   , 54.   , 13.   , 58.   , 23.   , 30.   ,
        57.   , 48.   , 40.   ,  2.   , 61.   , 64.   , 45.   , 39.   ,
        49.   , 26.   , 24.   , 11.   , 14.   , 20.   , 19.   , 56.   ,
        19.   , 42.   , 29.   , 65.   , 22.   ,  6.   , 10.   , 33.   ,
         4.   ,  1.   , 68.   , 63.   , 36.   , 21.   , 60.   , 31.   ,
        21.   , 47.   , 27.   , 17.   ,  3.   ,  4.   , 41.   , 65.   ,
        20.   , 66.   , 45.   , 55.   , 16.725]])

In [63]:
nation_decor_ranking=nation_decor_result_df[0][:-1]
nation_decor_runing_time=nation_decor_result_df[0][-1]

In [64]:
nation_decor_ranking.shape,nation_decor_runing_time,base_rankings_input.shape

((100,), 16.725, (8, 100))

In [65]:
nation_decor_perm=np.argsort(nation_decor_ranking)

In [66]:
nation_decor_ranking=np.argsort(nation_decor_perm)

In [67]:
nation_decor_kemeny_distance=kemeny_dist_paralle(base_rankings_input,nation_decor_ranking)
nation_decor_kemeny_distance

977.5

In [68]:
nation_decor_ranking_name=[idx_to_nation_dict[i] for i in nation_decor_perm]
nation_decor_ranking_name

['Germany',
 'Switzerland',
 'Netherlands',
 'United Kingdom',
 'Sweden',
 'United States',
 'Canada',
 'Japan',
 'France',
 'South Korea',
 'Australia',
 'Austria',
 'Belgium',
 'Italy',
 'Spain',
 'Poland',
 'Czech Republic',
 'Malaysia',
 'Israel',
 'Portugal',
 'China',
 'Chile',
 'Hungary',
 'Brazil',
 'United Arab Emirates',
 'Greece',
 'India',
 'Bulgaria',
 'Russia',
 'Saudi Arabia',
 'Argentina',
 'Romania',
 'Vietnam',
 'Turkey',
 'Thailand',
 'South Africa',
 'Mexico',
 'Indonesia',
 'Philippines',
 'Colombia',
 'Peru',
 'Ukraine',
 'Kazakhstan',
 'Serbia',
 'Morocco',
 'Tunisia',
 'Ecuador',
 'Sri Lanka',
 'Egypt',
 'Bangladesh',
 'Dominican Republic',
 'Iran',
 'Ghana',
 'Tanzania',
 'Algeria',
 'Kenya',
 'Belarus',
 'Jordan',
 'Pakistan',
 'Nepal',
 'Guatemala',
 'Bolivia',
 'Uzbekistan',
 'Angola',
 'Senegal',
 'Ivory Coast',
 'Honduras',
 'Azerbaijan',
 'Zambia',
 'Nigeria',
 'Cambodia',
 'Uganda',
 'Myanmar',
 'Papua New Guinea',
 'Iraq',
 'Laos',
 'Ethiopia',
 'Madaga

In [69]:
nation_fastcons_result_df=pd.read_csv("nation_data/nation_fastcons_result.csv").to_numpy()
nation_fastcons_result_df

array([[9.200000e+01, 4.400000e+01, 6.300000e+01, 2.900000e+01,
        8.000000e+00, 2.200000e+01, 7.400000e+01, 5.300000e+01,
        4.600000e+01, 1.400000e+01, 8.700000e+01, 6.500000e+01,
        2.000000e+01, 4.500000e+01, 8.200000e+01, 9.900000e+01,
        7.100000e+01, 7.700000e+01, 7.000000e+00, 9.600000e+01,
        2.600000e+01, 6.000000e+00, 3.900000e+01, 7.300000e+01,
        2.700000e+01, 6.100000e+01, 4.900000e+01, 4.200000e+01,
        5.700000e+01, 1.100000e+01, 2.000000e+00, 5.000000e+01,
        3.800000e+01, 6.800000e+01, 8.500000e+01, 9.400000e+01,
        7.500000e+01, 3.200000e+01, 1.500000e+01, 3.000000e+01,
        4.700000e+01, 6.600000e+01, 3.400000e+01, 1.300000e+01,
        6.900000e+01, 1.000000e+01, 6.400000e+01, 4.100000e+01,
        5.900000e+01, 7.600000e+01, 8.000000e+01, 8.900000e+01,
        1.900000e+01, 8.300000e+01, 2.100000e+01, 4.800000e+01,
        8.100000e+01, 5.800000e+01, 6.700000e+01, 5.000000e+00,
        9.500000e+01, 8.800000e+01, 5.40

In [70]:
nation_fastcons_ranking=nation_fastcons_result_df[0][:-1]
nation_fastcons_runing_time=nation_fastcons_result_df[0][-1]

In [71]:
nation_fastcons_ranking.shape,nation_fastcons_runing_time

((100,), 1430.883)

In [72]:
nation_fastcons_kemeny_distance=kemeny_dist_paralle(base_rankings_input,nation_fastcons_ranking)
nation_fastcons_kemeny_distance

994.75

In [73]:
nation_fastcons_perm=np.argsort(nation_fastcons_ranking)
nation_fastcons_ranking_name=[idx_to_nation_dict[i] for i in nation_fastcons_perm]
nation_fastcons_ranking_name

['United States',
 'Germany',
 'United Kingdom',
 'Switzerland',
 'Netherlands',
 'China',
 'Canada',
 'Australia',
 'Sweden',
 'Japan',
 'France',
 'South Korea',
 'Italy',
 'Belgium',
 'India',
 'Spain',
 'Poland',
 'Russia',
 'Malaysia',
 'Brazil',
 'Mexico',
 'Austria',
 'United Arab Emirates',
 'Saudi Arabia',
 'Turkey',
 'Chile',
 'Czech Republic',
 'Portugal',
 'Argentina',
 'Indonesia',
 'Vietnam',
 'Hungary',
 'Thailand',
 'Israel',
 'Philippines',
 'South Africa',
 'Romania',
 'Greece',
 'Colombia',
 'Peru',
 'Kazakhstan',
 'Egypt',
 'Ukraine',
 'Algeria',
 'Bulgaria',
 'Belarus',
 'Iran',
 'Morocco',
 'Ecuador',
 'Ghana',
 'Serbia',
 'Tunisia',
 'Bangladesh',
 'Nigeria',
 'Tanzania',
 'Pakistan',
 'Ethiopia',
 'Myanmar',
 'Kenya',
 'Sri Lanka',
 'Dominican Republic',
 'Uzbekistan',
 'Angola',
 'Jordan',
 'Bolivia',
 'Iraq',
 'Nepal',
 'Guatemala',
 'Ivory Coast',
 'Senegal',
 'Cambodia',
 'Zambia',
 'Cuba',
 'Azerbaijan',
 'Honduras',
 'Laos',
 'Cameroon',
 'Papua New Guinea